# Amazon — Baseline ANN (Plain MLP)
6 engineered features · No embeddings · Saves to `baseline_ANN/`

## Roadmap
```
STEP 1 — Imports & paths
STEP 2 — Load data
STEP 3 — Features & scale
STEP 4 — Dataset & DataLoader
STEP 5 — Model definition
STEP 6 — Train
STEP 7 — Evaluate
STEP 8 — Save results
```

---
## Step 1 — Imports & Paths

In [ ]:
import os,json,time,warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data      import Dataset,DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics       import mean_squared_error,mean_absolute_error

SEED=42; torch.manual_seed(SEED); np.random.seed(SEED)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

In [ ]:
DATA_DIR = r"C:\\Users\\Shivanshu Sirohi\\OneDrive\\Desktop\\Shivanshu\\White Paper\\Personalization\\amazon\\data"
OUT_DIR  = r"C:\\Users\\Shivanshu Sirohi\\OneDrive\\Desktop\\Shivanshu\\White Paper\\Personalization\\amazon\\baseline_ANN"
import os; os.makedirs(OUT_DIR, exist_ok=True)
print(f"Out dir: {OUT_DIR}")


---
## Step 2 — Load Data

In [ ]:
df_train = pd.read_csv(os.path.join(DATA_DIR, 'train_features.csv'))
df_val   = pd.read_csv(os.path.join(DATA_DIR, 'val_features.csv'))
df_test  = pd.read_csv(os.path.join(DATA_DIR, 'test_features.csv'))
print(f"Train : {df_train.shape}  Val : {df_val.shape}  Test : {df_test.shape}")


---
## Step 3 — Features & Scale

In [ ]:
FEATURE_COLS = [
    'user_avg_rating', 'user_rating_count',
    'product_avg_rating', 'product_rating_count', 'product_rating_std',
    'days_since_review'
]
TARGET = 'ratings'
print(f"Features : {len(FEATURE_COLS)}  Target : {TARGET}")


In [ ]:
scaler  = StandardScaler()
X_train = scaler.fit_transform(df_train[FEATURE_COLS])
X_val   = scaler.transform(df_val[FEATURE_COLS])
X_test  = scaler.transform(df_test[FEATURE_COLS])
y_train = df_train[TARGET].values
y_val   = df_val[TARGET].values
y_test  = df_test[TARGET].values
print(f"X_train : {X_train.shape}")


---
## Step 4 — Dataset & DataLoader

In [ ]:
class RatingsDataset(Dataset):
    def __init__(self,X,y):
        self.X=torch.tensor(X,dtype=torch.float32)
        self.y=torch.tensor(y,dtype=torch.float32)
    def __len__(self): return len(self.y)
    def __getitem__(self,idx): return self.X[idx],self.y[idx]

BATCH_SIZE=512
train_loader=DataLoader(RatingsDataset(X_train,y_train),batch_size=BATCH_SIZE,shuffle=True)
val_loader  =DataLoader(RatingsDataset(X_val,  y_val),  batch_size=BATCH_SIZE,shuffle=False)
test_loader =DataLoader(RatingsDataset(X_test, y_test), batch_size=BATCH_SIZE,shuffle=False)
X_b,y_b=next(iter(train_loader))
print(f'Batch — X: {X_b.shape}  y: {y_b.shape}')

---
## Step 5 — Model Definition
Input (6 features) → 64 → ReLU → 32 → ReLU → 1
Smaller network than before — appropriate for only 6 input features.

In [ ]:
class PlainMLP(nn.Module):
    def __init__(self,input_dim):
        super().__init__()
        self.net=nn.Sequential(
            nn.Linear(input_dim,64),nn.ReLU(),
            nn.Linear(64,32),nn.ReLU(),
            nn.Linear(32,1)
        )
    def forward(self,x): return self.net(x).squeeze(1)

model=PlainMLP(X_train.shape[1]).to(DEVICE)
print(model)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

---
## Step 6 — Train

In [ ]:
def run_epoch(model,loader,criterion,optimizer=None):
    model.train() if optimizer else model.eval()
    preds_all,labels_all=[],[]
    ctx=torch.enable_grad() if optimizer else torch.no_grad()
    with ctx:
        for X_b,y_b in loader:
            X_b,y_b=X_b.to(DEVICE),y_b.to(DEVICE)
            p=model(X_b); loss=criterion(p,y_b)
            if optimizer:
                optimizer.zero_grad(); loss.backward(); optimizer.step()
            preds_all.extend(p.detach().cpu().numpy())
            labels_all.extend(y_b.cpu().numpy())
    p=np.array(preds_all); l=np.array(labels_all)
    return (round(float(np.sqrt(mean_squared_error(l,p))),4),
            round(float(mean_absolute_error(l,p)),4))

In [ ]:
NUM_EPOCHS=20
criterion=nn.MSELoss()
optimizer=optim.Adam(model.parameters(),lr=0.001)
history={'train_rmse':[],'val_rmse':[]}

t0=time.perf_counter()
for epoch in range(NUM_EPOCHS):
    tr_rmse,_  =run_epoch(model,train_loader,criterion,optimizer)
    val_rmse,_ =run_epoch(model,val_loader,  criterion)
    history['train_rmse'].append(tr_rmse)
    history['val_rmse'].append(val_rmse)
    if (epoch+1)%5==0 or epoch==0:
        print(f'Epoch {epoch+1:>2}/{NUM_EPOCHS}  Train RMSE: {tr_rmse}  Val RMSE: {val_rmse}')
train_time=time.perf_counter()-t0
print(f'Training time: {train_time:.1f}s')

---
## Step 7 — Evaluate

In [ ]:
tr_rmse, tr_mae  =run_epoch(model,train_loader,criterion)
val_rmse,val_mae =run_epoch(model,val_loader,  criterion)
te_rmse, te_mae  =run_epoch(model,test_loader, criterion)

print('='*50)
print('ANN — Baseline Results')
print('='*50)
print(f'  Train      RMSE: {tr_rmse}   MAE: {tr_mae}')
print(f'  Validation RMSE: {val_rmse}   MAE: {val_mae}')
print(f'  Test       RMSE: {te_rmse}   MAE: {te_mae}')
print(f'  Train time : {train_time:.1f}s')
print(f'  Train-Test gap : {round(te_rmse-tr_rmse,4)}')

---
## Step 8 — Save Results

In [ ]:
result = {
    'model': 'ANN (Baseline)',
    'train_rmse': tr_rmse, 'val_rmse': val_rmse, 'test_rmse': te_rmse,
    'train_mae':  tr_mae,  'val_mae':  val_mae,  'test_mae':  te_mae,
    'train_time_s': round(train_time, 2),
}
pd.DataFrame([result]).to_csv(os.path.join(OUT_DIR, 'ann_results.csv'), index=False)
with open(os.path.join(OUT_DIR, 'ann_result.json'), 'w') as f:
    json.dump(result, f, indent=2)
print(f"Saved → {OUT_DIR}")
print(f"Test RMSE: {te_rmse}  Test MAE: {te_mae}")
